In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

df=pd.read_csv('/kaggle/input/datasets/ektanegi/spotifydata-19212020/data.csv')
df.head()


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
print("Rows and Columns:", df.shape)

df.info()

In [ ]:
df.head(10)
df.describe()

In [ ]:
df.isnull().sum()
df.fillna(df.mean(numeric_only=True), inplace=True)
print("Duplicate Rows:", df.duplicated().sum())

df.drop_duplicates(inplace=True)

In [ ]:
features = [
    'acousticness',
    'danceability',
    'energy',
    'instrumentalness',
    'liveness',
    'loudness',
    'speechiness',
    'tempo',
    'valence',
    'popularity'
]

selected_df = df[features]
selected_df.head()

selected_df.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['popularity'], bins=20)
plt.title("Popularity Distribution")
plt.xlabel("Popularity")
plt.ylabel("Songs")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

sns.heatmap(
    df[features].corr(),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [17]:
from sklearn.cluster import KMeans

X = df[features]

kmeans = KMeans(n_clusters=5, random_state=42)

df['cluster'] = kmeans.fit_predict(X)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(df[features])

In [34]:
df.to_csv('/kaggle/working/spotify_prepared.csv', index=False)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(1,11), wcss, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

In [22]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

df['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
df['cluster'].value_counts()

In [ ]:
cluster_summary = df.groupby('cluster')[features].mean()

print(cluster_summary)

In [25]:
cluster_names = {
    0: "classical/relaxing",
    1: "party/energetic",
    2: "romantic",
    3: "happy/feel-good",
    4: "rap"
}

In [26]:
df.to_csv(
    '/kaggle/working/spotify_clustered.csv',
    index=False
)

In [27]:
keyword_map = {

    # Cluster 0 : Classical / Relaxing
    "relax": 0,
    "relaxing": 0,
    "calm": 0,
    "peaceful": 0,
    "sleep": 0,
    "study": 0,
    "focus": 0,
    "meditation": 0,
    "classical": 0,
    "instrumental": 0,
    "soft": 0,

    # Cluster 1 : Party / Energetic
    "party": 1,
    "dance": 1,
    "energetic": 1,
    "energy": 1,
    "workout": 1,
    "gym": 1,
    "running": 1,
    "exercise": 1,
    "celebration": 1,
    "fun": 1,
    "festival": 1,

    # Cluster 2 : Romantic
    "romantic": 2,
    "romance": 2,
    "love": 2,
    "couple": 2,
    "date": 2,
    "relationship": 2,
    "heart": 2,
    "valentine": 2,
    "crush": 2,
    "emotion": 2,
    "affection": 2,

    # Cluster 3 : Happy / Feel-Good
    "happy": 3,
    "feelgood": 3,
    "feel-good": 3,
    "positive": 3,
    "cheerful": 3,
    "joy": 3,
    "smile": 3,
    "uplifting": 3,
    "good mood": 3,
    "motivational": 3,
    "fresh": 3,

    # Cluster 4 : Rap
    "rap": 4,
    "hiphop": 4,
    "hip-hop": 4,
    "trap": 4,
    "freestyle": 4,
    "bars": 4,
    "rapping": 4,
    "drill": 4,
    "beat": 4,
    "street": 4
}

In [ ]:
user_input = input("Enter mood: ").lower()

cluster = keyword_map[user_input]

recommendations = df[df['cluster'] == cluster]

recommendations = recommendations.sort_values(
    by='popularity',
    ascending=False
)

print(recommendations[['name', 'artists', 'popularity']].head(10))

In [ ]:
#timeRecommendation

from datetime import datetime

current_hour = datetime.now().hour

print(current_hour)

def get_time_of_day():

    hour = datetime.now().hour

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"

time_pref = get_time_of_day()

print(time_pref)


In [32]:
time_cluster = {
    "Morning": 0,
    "Afternoon": 1,
    "Evening": 2,
    "Night": 3,
    "Late Evening":4
}

In [ ]:
def recommend_songs():

    current_time = get_time_of_day()

    cluster = time_cluster[current_time]

    recommendations = df[df['cluster'] == cluster]

    recommendations = recommendations.sort_values(
        by='popularity',
        ascending=False
    )

    return recommendations[
        ['name', 'artists', 'popularity']
    ].head(10)

songs = recommend_songs()

print("Recommended Songs:\n")
print(songs)